### Reading data from source

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df =spark.sql('select * from databricks_cat_1234.silver.customers_silver order by customer_id')
df.display()

### Remove duplicates

In [0]:
df=df.dropDuplicates(subset=['customer_id'])
df.limit(10).display()


### Dividing New vs Old records

### Surrogate key - All the values

In [0]:
df=df.withColumn('DimCustomerKey',monotonically_increasing_id()+lit(1))

In [0]:
df.display()

In [0]:
init_load_flag= int(dbutils.widgets.get('init_load_flag'))

In [0]:
if init_load_flag == 0:
    df_old = spark.sql('''select DimCustomerKey,customer_id,create_date,update_date from databricks_cat_1234.gold.DimCustomers''')
else:
    df_old = spark.sql('''select 0 DimCustomerKey,0 customer_id,0 create_date,0 update_date from databricks_cat_1234.silver.customers_silver where 1=0''')

In [0]:
df_old.display()

### Renaming column name

In [0]:
df_old= df_old.withColumnRenamed("DimCustomerKey",'old_DimCustomerKey')\
            .withColumnRenamed("customer_id",'old_customer_id')\
            .withColumnRenamed("create_date",'old_create_date')\
            .withColumnRenamed("update_date",'old_update_date')



In [0]:
df_join = df.join(df_old ,df.customer_id==df_old.old_customer_id, 'left')
df_join.display()

## ### Seperating Old vs New records

In [0]:
df_new = df_join.filter(df_join['old_DimCustomerKey'].isNull())
df_new.display()

In [0]:
df_old = df_join.filter(df_join['old_DimCustomerKey'].isNotNull())
df_old.display()

# Preparing df_old

In [0]:
# Droppping all the columns which are not required
df_old = df_old.drop('DimCustomerKey','old_customer_id','old_update_date')

# Renaming old_DimCustomerKey column to 'DimCustomerKey'
df_old = df_old.withColumnRenamed('old_DimCustomerKey','DimCustomerKey')

# Renaming old_create_date column to 'create_date'
df_old= df_old.withColumnRenamed('old_create_date','create_date')
df_old= df_old.withColumn('create_date',to_timestamp(col('create_date')))

# Recreating Update date column with current timestamp
df_old=df_old.withColumn('update_date',current_timestamp())



In [0]:
df_old.display()

# Preparing df_new

In [0]:
# Droppping all the columns which are not required
df_new =df_new.drop('old_DimCustomerKey','old_customer_id','old_update_date','old_create_date')

# Renaming old_create_date,old_update_date columns with current timestamp
df_new=df_new.withColumn('create_date',current_timestamp())
df_new=df_new.withColumn('update_date',current_timestamp())


In [0]:
df_new.display()

# Adding Max Surrogate key

In [0]:
if init_load_flag ==1:
    max_surrogate_key=0
else:
    df_maxsur=spark.sql('select max(DimCustomerKey) as max_surrogate_key from databricks_cat_1234.gold.DimCustomers')
    #### ## Converting max_surr to max_surrogate_key variable
    max_surrogate_key = df_maxsur.collect()[0]['max_surrogate_key']

In [0]:
df_new.display()

In [0]:
df_new = df_new.withColumn('DimCustomerkey',lit(max_surrogate_key)+col('DimCustomerkey'))

In [0]:
df_new.display()

**Union of df_old and df_new**

In [0]:
df_old.limit(2).display()

In [0]:
df_new.limit(2).display()

In [0]:
df_final= df_new.unionByName(df_old)


In [0]:
df_final.display()

## SCD TYPE - 1

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists('databricks_cat_1234.gold.DimCustomers'):
    print('hellow12')
    dlt_object= DeltaTable.forPath(spark,'abfss://gold@datalakeete1.dfs.core.windows.net/DimCustomers')
    dlt_object.alias('trg').merge(df_final.alias('src'),'trg.DimCustomerKey=src.DimCustomerKey')\
            .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll()\
            .execute()
else:
    print('hellow')
    df_final.write.mode('overwrite')\
        .option('path','abfss://gold@datalakeete1.dfs.core.windows.net/DimCustomers')\
        .saveAsTable('databricks_cat_1234.gold.DimCustomers')
    
    


In [0]:
%sql
select * from databricks_cat_1234.gold.DimCustomers;